In [65]:
import os
from osgeo import gdal
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
import geopandas as gpd
import sys
from shapely.geometry import Polygon
import math
import gc
from sqlalchemy import create_engine, text
import dask.array as da
import re
import glob

In [81]:
def get_raster_file_list(path):
    """Get a list of the raster files inside the folder"""
    File_list = [] #f for f in os.listdir(path) if os.isfile(mypath,f)
    for file in os.listdir(path):
        if file.endswith(".tif") or file.endswith(".tiff"):
            if file not in File_list:
                File_list.append(os.path.join(path,file))
        else:
            pass
    return File_list

def load_vector_layer(db_name, user, password, host, port, table_name, schema='public', geom_col='geom'):
    """
    Connects to a PostGIS-enabled PostgreSQL database and loads a vector layer as a GeoDataFrame.
    
    Parameters:
    - db_name (str): Name of the PostgreSQL database.
    - user (str): Database username.
    - password (str): Database password.
    - host (str): Host address (e.g., 'localhost' or IP).
    - port (int): Port number (e.g., 5432).
    - table_name (str): Name of the table (vector layer) to load.
    - schema (str): Optional. Database schema containing the table (default is 'public').

    Returns:
    - gpd.GeoDataFrame: A GeoDataFrame containing the vector layer.
    """
    try:
        # Use pg8000 (pure Python driver)
        conn_str = f"postgresql+pg8000://{user}:{password}@{host}:{port}/{db_name}"
        engine = create_engine(conn_str)

        sql = text(f"SELECT * FROM {schema}.{table_name}")

        # Open a connection explicitly (SQLAlchemy 2.x requirement)
        with engine.connect() as conn:
            gdf = gpd.read_postgis(sql, conn, geom_col=geom_col)
        
        print(f"Successfully loaded {table_name} ({len(gdf)} features)")
        return gdf

    except Exception as e:
        print(f"Error loading vector layer: {e}")
        return None

def filter_countries(gdf_gadm, countries):

    available_countries = gdf_gadm['country_n'].unique().tolist()
    selected_countries = countries.copy()

    # Check for missing countries
    missing_countries = [c for c in selected_countries if c not in available_countries]

    if missing_countries:
        return print("Warning: These countries are not in the DataFrame:", missing_countries)
    
    else:
        gdf_gadm_countries = gdf_gadm[gdf_gadm['country_n'].isin(countries)]
        return gdf_gadm_countries

def get_geometry_grid(geodataframe, epsg):
    """
    get_geometry_grid creates a defined grid of the input territory extension.
    :geodataframe: the territory vector file ina gdf format.
    :epsg: the defined epsg of vector georeferenced data.
    :return: the grid as a geodataframe
    """
    
    #get the bounds of the territory
    xmin, ymin, xmax, ymax = geodataframe.total_bounds
    # define the size of the grid in degrees
    length = 5
    wide = 5
    # set the cols and rows
    cols = list(np.arange(xmin, xmax + wide, wide))
    rows = list(np.arange(ymin, ymax + length, length))
    # create a list of all the polygons containing the grid.
    polygons = []
    for x in cols[:-1]:
        for y in rows[:-1]:
            polygons.append(Polygon([(x,y), (x+wide, y), (x+wide, y+length), (x, y+length)]))
    # transform the polygon list into a Geoseries or Geodataframe.
    # grid = gpd.GeoSeries({'geometry':MultiPolygon(polygons)})
    grid = gpd.GeoDataFrame({'geometry':polygons}, crs=epsg)
    return grid

def area_of_pixel(pixel_size, center_lat):
    """
    area_of_pixel calculates the area, in hectares, of a wgs84 square raster
    tile given its latitude and side-length.
    This function is adapted from https://gis.stackexchange.com/a/288034.

    :param pixel_size: is the length of the pixel side in degrees.
    :param center_lat: is the latitude of the center of the pixel. This value
    +/- half the `pixel-size` must not exceed 90/-90 degrees latitude or an
    invalid area will be calculated.
    :return: the rel area in hectares of a square pixel of side length
    `pixel_size` whose center is at latitude `center_lat`.
    """

    a = 6378137  # meters
    b = 6356752.3142  # meters
    e = math.sqrt(1 - (b/a)**2)
    area_list = []
    for f in [center_lat+pixel_size/2, center_lat-pixel_size/2]:
        zm = 1 - e*math.sin(math.radians(f))
        zp = 1 + e*math.sin(math.radians(f))
        area_list.append(
            math.pi * b**2 * (
                math.log(zp/zm) / (2*e) +
                math.sin(math.radians(f)) / (zp*zm)))
    return (pixel_size / 360. * (area_list[0] - area_list[1])) * np.power(10.0,-4)

def aggregate_one_region(out_image, out_transform, pixel_size, width_0, height_0, width_1, height_1):
    """
    aggregate_one_region performs the aggregation of the density observable in a
    given region supplied in the form of a masked raster of the observable.

    :param out_image: the masked raster layer to aggregate.
    :param out_transform: the Affine of the raster layer.
    :param pixel_size: the side length in degrees of each square raster tile.
    :param width_0: the starting value position of the width array
    :param height_0: the starting value position of the height array
    :param width_1: the end value position of the width array
    :param height_1: the end value position of the height array
    :return: the aggregated value of the observable in the specified region.
    """

    # Create a matrix of coordinates based on tile number.
    cols, rows = np.meshgrid(np.arange(width_0, width_1), np.arange(height_0, height_1))

    # Transform the tile number coordinates to real coordinates and extract only
    # latitude information.
    ys = rasterio.transform.xy(out_transform, rows, cols)[1] # [0] is xs
    # latitudes = np.array(ys) # Cast the list of arrays to a 2D array for computational convenience.
    ys = cols = rows = None #empty the memory

    # Iterate over the latitudes matrix, calculate the area of each tile, and
    # store it in the real_raster_areas array.
    # real_raster_areas = np.empty(np.shape(latitudes))
    # for i, latitude_array in enumerate(latitudes):
    #     for j, latitude in enumerate(latitude_array):
    #         real_raster_areas[i,j] = area_of_pixel(pixel_size, latitude)

    # Calculate the total value in each tile: density * area = observable value
    # in the area.
    # value = real_raster_areas * out_image[0,height_0:height_1,width_0:width_1] #I don't think np.transpose() is necesary

    # Alternative
    value = out_image[0,height_0:height_1,width_0:width_1]

    out_image = None #empty the memory
    # Sum all the carbon stock values in the country treating NaNs as 0.0.
    aggregated_value = np.nansum(value)

    return aggregated_value

def aggregate_one_region_dask(out_image, out_transform, pixel_size, width_0, height_0, width_1, height_1):

    # ---- 1. Compute one latitude per row ----
    # Create an array of row indices for the selected window.
    rows = np.arange(height_0, height_1)
    cols = np.full(rows.shape, width_0)

    # rasterio.transform.xy returns (xs, ys) for each (row, col) pair.
    # We only extract the latitude = ys.
    # ys is a list/array of latitudes for each row.
    row_lats = np.array(rasterio.transform.xy(out_transform, rows, cols)[1])

    # ---- 2. Compute area per row ----
    # Since each row has a constant latitude, we compute ONE area per row:
    row_areas = np.array([area_of_pixel(pixel_size, lat) for lat in row_lats])

    # ---- 3. Broadcast using Dask ----

    # row_areas has shape: (num_rows,)
    # We want to broadcast it against the raster slice (num_rows, num_cols).
    # So we reshape it to (num_rows, 1) using [:, None].
    # We transform the array [1, 2, 3] → [[1], (column vector)
    #                                     [2],
    #                                     [3]]
    area_matrix = da.from_array(row_areas[:, None], chunks=(512, 1))

    density = da.from_array(
                out_image[0, # band of the image
                height_0:height_1,
                width_0:width_1],
                chunks=(512, 512) # splits the image into 512×512 blocks
                )

    # ---- 4. Multiply + sum ----
    values = density * area_matrix
    sum_result = da.nansum(values).compute() # sum everything, treating NaN as 0
    return sum_result

def aggregate_density_observable(raster_files_list, region_polygons, temp_export_path):
    """
    aggregate_density_observable aggregates the density observable for all the
    raster files specified and inside the specified regions. The result of the
    aggregation is returned as a table and the result for each year/raster is
    progressively exported in CSV format.

    :param raster_files_list: a list containing the addresses of all the raster
    files that store the observable's data for each year.
    :param region_polygons: a GeoDataFrame storing the polygons corresponding to
    each region used for the aggregation.
    :param temp_export_path: path to export the temporary results.
    :return: a DataFrame storing the aggregated vegetation carbon stocks at the
    region level for each year.
    """

    # Final DataFrame will store the aggregated carbon stocks for each country and each year.
    aggregated_df = pd.DataFrame([])

    for file in raster_files_list[:]: # [10:]
        # Get the string values as a list. 
        # file_year = re.findall(r'\d+', file)[3]
        file_year = "2020" # It has to be a str
        # file_year = file.split('_')[-2]
        try:
        #    landcover_class = re.findall(r'\d+', file)[3]
           landcover_class = os.path.basename(file).split("_merged_total_")[1].replace(".tif", "")
        #    landcover_class = file.split('_')[-1].replace(".tif", "")
        except:
            landcover_class = False
            
        """For coastal"""
        product = os.path.basename(file).split("_merged")[0]

        print("Processing file {} corresponding to year {}.".format(file, file_year))

        # This list will store the results from the aggregation.
        aggregated_value_list = []
        territory_list = []

        with rasterio.open(file) as raster_file: # Load the raster file.

            gt = raster_file.transform # Get all the raster properties on a list.
            pixel_size = gt[0] # X size is stored in position 0, Y size is stored in position 4.

            error_countries_id = [] # Create a list for all encountered possible errors
            
            # create both counter if we want to tmp export resuls at reaching n number of countries
            # country_counter = 1
            # country_counter_iterator = 1

            # Take the geometry name
            geometry_name = region_polygons.geometry.name

            for row_index, row in region_polygons.iterrows(): # gdf.loc[0:1].iterrows(): / gdf.loc(axis=0)[0:1] / df[df['column'].isin([1,2])]
                # total_aggregated_value = 0
                try:
                    # Iterate over the country polygons to progressively calculate the total carbon stock in each one of them.

                    """"OPTION 1"""
                    geo_tile_row = gpd.GeoSeries(row[geometry_name])

                    # If it does not overlap
                    out_image, out_transform = mask(raster_file, geo_tile_row, crop=True, filled=True, nodata=np.nan) # make sure the nodata goes to nan.
                    # print("out_image memory usage {}".format(out_image.nbytes / np.power(10.0,9))) # check memory for testing

                    if np.isnan(out_image).all():
                        aggregated_value = 0
                        # continue
                    else:
                        # Obtain the number of tiles in both directions.
                        height = out_image.shape[1]
                        width  = out_image.shape[2]
                        #calculate the aggregated value.
                        aggregated_value = aggregate_one_region(out_image, out_transform, pixel_size, 0, 0, width, height)

                    # aggregated_value = aggregate_one_region_dask(out_image, out_transform, pixel_size, 0, 0, width, height)

                    # print("aggregated value memory usage {}".format(sys.getsizeof(aggregated_value))) # check memory for testing
                    # accumulate the results.
                    # total_aggregated_value += aggregated_value 
                    # clean memory
                    out_image = None # this cleans the memory to 16 bits
                    out_transform = None

                    #force cleaning the memory
                    gc.collect()
                    print("{} out of {} of country {} with values {}".format(row_index + 1, len(region_polygons), row["gid_n"], aggregated_value))

                    """"OPTION 2"""

                    # geodf_row = gpd.GeoDataFrame(geometry=gpd.GeoSeries(row[geometry_name]), crs=4326) # This is the country's polygon geometry df.
                    # # geo_row = gpd.GeoSeries(row['geometry']) # This is the country's polygon geometry.

                    # # create a grid of the territory with the coresponding EPSG
                    # grid = get_geometry_grid(geodf_row, "EPSG:4326")
                    # print("grid memory usage {}".format(sys.getsizeof(grid)))
                    # # adjust the grid to the shape of the territory
                    # region_grid = grid.overlay(geodf_row, how="intersection").to_crs(epsg='4326') # this operation requires both inputs to be gdf.
                    # print("region_grid memory usage {}".format(sys.getsizeof(region_grid)))
                    # # region_grid.to_file('region_grid.shp') # check memory for testing
                    # # iterate over each tile and accumulate the value

                    # total_tiles = int(len(region_grid))
                    # for row_index, tile_row in region_grid.iterrows():
                    #     # print a process index.
                    #     print("{} out of {} of country {}".format(row_index + 1, total_tiles, row["gid_n"]))
                    #     geo_tile_row = gpd.GeoSeries(tile_row['geometry'])
                    #     # repeat the masking process with the tile.
                    #     try:
                    #         # If it does not overlap
                    #         out_image, out_transform = rasterio.mask.mask(raster_file, geo_tile_row, crop=True, filled=True, nodata=np.nan) # make sure the nodata goes to nan.
                    #     # print("out_image memory usage {}".format(out_image.nbytes / np.power(10.0,9))) # check memory for testing
                    #     except Exception as e:
                    #         # print(f"The tile {tile_row} has errors: ", e)
                    #         continue

                    #     if np.isnan(out_image).all():
                    #         continue

                    #     # Obtain the number of tiles in both directions.
                    #     height = out_image.shape[1]
                    #     width  = out_image.shape[2]
                    #     #calculate the aggregated value.
                    #     aggregated_value = aggregate_one_region(out_image, out_transform, pixel_size, 0, 0, width, height)

                    #     # aggregated_value = aggregate_one_region_dask(out_image, out_transform, pixel_size, 0, 0, width, height)

                    #     # print("aggregated value memory usage {}".format(sys.getsizeof(aggregated_value))) # check memory for testing
                    #     # accumulate the results.
                    #     total_aggregated_value += aggregated_value 
                    #     # clean memory
                    #     out_image = None # this cleans the memory to 16 bits
                    #     out_transform = None

                    #     #force cleaning the memory
                    #     gc.collect()
                        
                except Exception as e:
                    # In case there is an error on the process, a value of -9999.0 will be appended
                    print("the country {} with index {} has errors: {}".format(row["gid_n"], row["id"], e) )
                    error_countries_id.append(row["id"])
                    aggregated_value = -9999.0 

                
                # Add the aggregated stock to the list.
                # aggregated_value_list.append(total_aggregated_value)
                aggregated_value_list.append(aggregated_value)
                territory_list.append(row["territory1"])

                print("the country {} with index {} is finished with total carbon of: {}".format(row["gid_n"], row["id"], aggregated_value))
                
                """this part is to create additional internal temporal results"""
                # country_counter += 1
                # if country_counter_iterator < 26: # + 1
                #     country_counter_iterator += 1
                # else:
                # if not landcover_class:
                #     aggregated_observable = pd.DataFrame(row["ADM0_NAME"], aggregated_value, columns = ["country", file_year])
                #     print(aggregated_observable.head())
                #     # Export the temporary results from curent year.
                #     aggregated_observable.to_csv(temp_export_path + "_" + row["ADM0_NAME"] + "_" + str(file_year) + ".csv")
                # else:
                #     aggregated_observable = pd.DataFrame(row["ADM0_NAME"], aggregated_value, columns = ["country", file_year + "_" + landcover_class])
                #     aggregated_observable.to_csv(temp_export_path + "_" + row["ADM0_NAME"] + "_" + str(file_year) + "_" + str(landcover_class) + ".csv")

                    # country_counter_iterator = 1

        print("Finished calculating {}.".format(file_year))
        if error_countries_id:
            print("countries id with error: ", error_countries_id)

        # Transform the list to a DataFrame using the year as header.
        if not landcover_class:
            aggregated_observable = pd.DataFrame(aggregated_value_list, columns = [file_year])
            # Export the temporary results from curent year.
            aggregated_observable.to_csv(temp_export_path + "_" + str(file_year) + ".csv")
        else:
            # product temporary
            # aggregated_observable = pd.DataFrame(aggregated_value_list, columns = [product + "_" + file_year + "_" + landcover_class])

            col_name = product + "_" + file_year + "_" + landcover_class
            aggregated_observable = pd.DataFrame({"country": territory_list, col_name: aggregated_value_list})

            # aggregated_observable.to_csv(os.path.join(temp_export_path, row["gid_n"] + "_" + str(file_year) + "_" + str(landcover_class) + ".csv"))
            aggregated_observable.to_csv(os.path.join(temp_export_path, os.path.basename(file).replace(".tif","") + "_1.csv"))
            
        # Merge this year's results with the final, multi-year DataFrame.
        aggregated_df = pd.merge(aggregated_df, aggregated_observable, how='outer', left_index = True, right_index=True)

    # aggregated_df.to_csv(os.path.join(temp_export_path, row["gid_n"] + "_total_" + str(file_year) + ".csv"))
    aggregated_df.to_csv(os.path.join(temp_export_path, "total_" + str(file_year) + "_1.csv"))

    return aggregated_df


In [67]:
"""Load the input"""
gdf_reference = load_vector_layer(
    db_name='geoserver',
    user='geoserver',
    password='geoserver',
    host='192.168.250.100',
    port=5555,
    table_name='global_osm_coastline_6km_buffer_country_split_4326',
    schema='public'
)

Successfully loaded global_osm_coastline_6km_buffer_country_split_4326 (6928 features)


In [68]:
gdf_reference.head()

,id,geom,gid,tile_index_1d,tile_index_3d,territory1,iso_sov1
0,1,"MULTIPOLYGON (((-68.74533 -56.42005, -68.74500...",1,11992,1358,Chile,CHL
1,2,"MULTIPOLYGON (((-67.24486 -56.00141, -67.24706...",2,11993,1358,Chile,CHL
2,16,"MULTIPOLYGON (((-66.98173 -54.97317, -66.98100...",21,12714,1358,Chile,CHL
3,17,"MULTIPOLYGON (((-74.00467 -53.23788, -74.00470...",33,13066,1476,Chile,CHL
4,18,"MULTIPOLYGON (((-73.25807 -53.99607, -73.25803...",34,13067,1476,Chile,CHL


In [69]:
gdf_reference_2 = gdf_reference.rename(columns={
    "iso_sov1": "gid_n"
    # "geom": "geometry"
})

In [70]:
gdf_reference_2.head()

,id,geom,gid,tile_index_1d,tile_index_3d,territory1,gid_n
0,1,"MULTIPOLYGON (((-68.74533 -56.42005, -68.74500...",1,11992,1358,Chile,CHL
1,2,"MULTIPOLYGON (((-67.24486 -56.00141, -67.24706...",2,11993,1358,Chile,CHL
2,16,"MULTIPOLYGON (((-66.98173 -54.97317, -66.98100...",21,12714,1358,Chile,CHL
3,17,"MULTIPOLYGON (((-74.00467 -53.23788, -74.00470...",33,13066,1476,Chile,CHL
4,18,"MULTIPOLYGON (((-73.25807 -53.99607, -73.25803...",34,13067,1476,Chile,CHL


In [ ]:
"""Filter the country"""
countries = ["Uganda"]
gdf_gadm_countries = filter_countries(gdf_reference, countries)
# Export it to load it instantly in the process
gdf_gadm_countries.to_file("uganda_gadm_countries.shp")

In [ ]:
"""Load the country"""
gdf_gadm_countries = gpd.read_file("Y:\z_resources\im-data-global-landcover\lc_analysis\output_fin_boundary.shp")
gdf_gadm_countries.head()

In [74]:
raster_files_path = r"Y:\z_resources\un_gbf\01_aggregation_phase\03_masked_outputs\coastal_protection_02"
temp_export_path = r"Y:\z_resources\un_gbf\01_aggregation_phase\04_tables\02_second_filter"

raster_files_list = get_raster_file_list(raster_files_path)

In [80]:
for file in raster_files_list[0:1]: # [10:]
    # Get the string values as a list. 
    # file_year = re.findall(r'\d+', file)[3]
    # file_year = file.split('_')[-2]
    file_year = os.path.basename(file).split("_merged_total_")[1].replace(".tif", "")
    product = os.path.basename(file).split("_merged")[0]
print(product)  # cp_supply
print(file_year[:])

cp_demand
183_highvulnerability


In [ ]:
aggregate_density_observable(raster_files_list, gdf_reference_2, temp_export_path)

Processing file Y:\z_resources\un_gbf\01_aggregation_phase\03_masked_outputs\coastal_protection_02\cp_demand_merged_total_183_highvulnerability.tif corresponding to year 2020.
1 out of 6928 of country CHL with values 0
the country CHL with index 1 is finished with total carbon of: 0
2 out of 6928 of country CHL with values 0
the country CHL with index 2 is finished with total carbon of: 0
3 out of 6928 of country CHL with values 0
the country CHL with index 16 is finished with total carbon of: 0
4 out of 6928 of country CHL with values 0
the country CHL with index 17 is finished with total carbon of: 0
5 out of 6928 of country CHL with values 0.0
the country CHL with index 18 is finished with total carbon of: 0.0
6 out of 6928 of country CHL with values 0.0
the country CHL with index 19 is finished with total carbon of: 0.0
7 out of 6928 of country CHL with values 0
the country CHL with index 20 is finished with total carbon of: 0
8 out of 6928 of country CHL with values 0.056104999035

# Tables building

In [46]:
table_csv = r"Y:\z_resources\un_gbf\01_aggregation_phase\04_tables\total_2020.csv"
table_df = pd.read_csv(table_csv)

In [ ]:
table_df["country"] = gdf_reference_2["territory1"].values

In [48]:
gdf_reference_2["territory1"].nunique()

253

In [59]:
path = r"Y:\z_resources\un_gbf\01_aggregation_phase\04_tables\*.csv"
results = []

for file in glob.glob(path):
    df = pd.read_csv(file)
    results.append({
        "file": os.path.basename(file),
        "num_rows": len(df)
    })

summary = pd.DataFrame(results)
print(summary)

                                     file  num_rows
0          cp_demand_merged_total_1_1.csv      1533
1        cp_demand_merged_total_1_183.csv      5234
2      cp_demand_merged_total_1_183_1.csv      6928
3        cp_demand_merged_total_1_185.csv      5037
4      cp_demand_merged_total_1_185_1.csv      6928
5        cp_demand_merged_total_1_1_1.csv      6928
6          cp_demand_merged_total_1_6.csv      1130
7        cp_demand_merged_total_1_6_1.csv      6928
8          cp_supply_merged_total_1_1.csv         0
9        cp_supply_merged_total_1_183.csv      5449
10     cp_supply_merged_total_1_183_1.csv      6928
11       cp_supply_merged_total_1_185.csv      5245
12     cp_supply_merged_total_1_185_1.csv      6928
13       cp_supply_merged_total_1_1_1.csv      6928
14         cp_supply_merged_total_1_6.csv      1173
15       cp_supply_merged_total_1_6_1.csv      6928
16      cp_total_new_merged_total_1_1.csv      1537
17    cp_total_new_merged_total_1_183.csv      4752
18  cp_total

In [64]:

from functools import reduce
all_summaries = []

for file in glob.glob(path):
    filename = os.path.basename(file)
    
    if filename == "00_total_2020_1.csv":
        continue
        
    # FIX: Use index_col=0 if the first column is just the row numbers (0, 1, 2...)
    # Or read it normally and drop it manually:
    df = pd.read_csv(file)
    
    # Drop columns that start with "Unnamed"
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    
    # 1. Aggregate this file
    df_sum = df.groupby("country").sum(numeric_only=True).reset_index()
    
    # 2. Rename columns (except 'country') to include filename 
    suffix = filename.replace(".csv", "")
    df_sum = df_sum.rename(columns=lambda x: f"{x}_{suffix}" if x != 'country' else x)
    
    all_summaries.append(df_sum)

# 3. Merge all dataframes on 'country'
if all_summaries:
    final_master_df = reduce(lambda left, right: pd.merge(left, right, on='country', how='outer'), all_summaries)
    final_master_df = final_master_df.fillna(0)
    
    print(final_master_df)
else:
    print("No data processed.")

                               country  \
0    Abu musa, Greater and Lesser Tunb   
1                               Alaska   
2                              Albania   
3                              Algeria   
4                    Alhucemas Islands   
..                                 ...   
248                            Vietnam   
249              Wake Island / Enenkio   
250                  Wallis and Futuna   
251                     Western Sahara   
252                              Yemen   

     cp_demand_2020_183_cp_demand_merged_total_1_183_1  \
0                                             3.502595   
1                                           429.852431   
2                                            51.603720   
3                                            62.987012   
4                                             0.000000   
..                                                 ...   
248                                        7800.828453   
249                            

In [ ]:
# 1. Stack all dataframes into a single one
combined_df = pd.concat(all_dataframes, ignore_index=True)